# BNOT quickstart (Colab)

This notebook bootstraps a Colab runtime, clones `BNOT_new` if needed, updates it to the latest `main`, reuses the repo-local environment and build when possible, and runs `ibnot_new_cli` through the Python wrapper interface.

Run the notebook from top to bottom. The bootstrap cell is designed to be idempotent inside the same Colab runtime.


## 1. Bootstrap the Colab runtime

This cell installs missing system tools only if needed, installs Miniforge only if needed, clones or updates the repository, recreates the repo-local Conda environment only when `environment.yml` changes, and rebuilds the CLI only when the `ibnot_cli` tree changes.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/Musialski-Research-Group/BNOT_new.git"
REPO_ROOT = Path("/content/BNOT_new")
MINIFORGE_DIR = Path("/usr/local/miniforge")
CONDA_EXE = MINIFORGE_DIR / "bin" / "conda"
ENV_PREFIX = REPO_ROOT / ".conda" / "ibnot_cli"
BUILD_DIR = REPO_ROOT / "ibnot_cli" / "build-colab"
CLI_BIN = BUILD_DIR / "ibnot_new_cli"
STATE_DIR = REPO_ROOT / ".colab_state"
ENV_STAMP = STATE_DIR / "environment_tree.txt"
BUILD_STAMP = STATE_DIR / "build_tree.txt"

needed_tools = []
if shutil.which("gs") is None:
    needed_tools.append("ghostscript")
if shutil.which("git") is None:
    needed_tools.append("git")
if shutil.which("wget") is None:
    needed_tools.append("wget")

if needed_tools:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", *needed_tools], check=True)
    print(f"Installed system packages: {needed_tools}")
else:
    print("System tools already present; skipping apt install.")

if not CONDA_EXE.exists():
    installer = Path("/tmp/Miniforge3.sh")
    subprocess.run([
        "wget",
        "-O",
        str(installer),
        "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
    ], check=True)
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)

if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    print(f"Cloned repo into {REPO_ROOT}")
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "reset", "--hard", "origin/main"], check=True)
    print(f"Updated existing clone at {REPO_ROOT} to origin/main")

STATE_DIR.mkdir(parents=True, exist_ok=True)
env_tree = subprocess.check_output([
    "git", "-C", str(REPO_ROOT), "rev-parse", "HEAD:ibnot_cli/environment.yml"
], text=True).strip()
build_tree = subprocess.check_output([
    "git", "-C", str(REPO_ROOT), "rev-parse", "HEAD:ibnot_cli"
], text=True).strip()

env = os.environ.copy()
env["CONDA_EXE"] = str(CONDA_EXE)
env_needs_update = (
    not (ENV_PREFIX / "conda-meta").exists()
    or not ENV_STAMP.exists()
    or ENV_STAMP.read_text().strip() != env_tree
)
if env_needs_update:
    subprocess.run(["bash", "./setup_env.sh"], cwd=REPO_ROOT / "ibnot_cli", env=env, check=True)
    ENV_STAMP.write_text(env_tree + "\n")
    print(f"Prepared repo-local env at {ENV_PREFIX}")
else:
    print(f"Reusing existing env at {ENV_PREFIX}")

build_needs_update = (
    not CLI_BIN.exists()
    or not BUILD_STAMP.exists()
    or BUILD_STAMP.read_text().strip() != build_tree
)
if build_needs_update:
    subprocess.run([
        str(CONDA_EXE),
        "run",
        "--prefix",
        str(ENV_PREFIX),
        "cmake",
        "-S",
        str(REPO_ROOT / "ibnot_cli"),
        "-B",
        str(BUILD_DIR),
        "-G",
        "Ninja",
        f"-DCMAKE_PREFIX_PATH={ENV_PREFIX}",
    ], check=True)
    subprocess.run([
        str(CONDA_EXE),
        "run",
        "--prefix",
        str(ENV_PREFIX),
        "cmake",
        "--build",
        str(BUILD_DIR),
    ], check=True)
    BUILD_STAMP.write_text(build_tree + "\n")
    print(f"Built CLI at {CLI_BIN}")
else:
    print(f"Reusing existing CLI build at {CLI_BIN}")

print(f"Repo root: {REPO_ROOT}")
print(f"Conda: {CONDA_EXE}")
print(f"Build dir: {BUILD_DIR}")


## 2. Load the Python interface and define shared settings

This cell loads the wrapper directly from the cloned repository source, constructs a reusable request builder, and defines the common image, render, and timing settings for all examples.


In [ ]:
from pathlib import Path
import importlib.util
import inspect
import shutil

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

REPO_ROOT = Path("/content/BNOT_new")
API_PATH = REPO_ROOT / "python" / "src" / "ibnot_cli_wrapper" / "api.py"
spec = importlib.util.spec_from_file_location("ibnot_cli_wrapper_repo_api", API_PATH)
ibnot_api = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(ibnot_api)

make_linear_ramp = ibnot_api.make_linear_ramp
make_sine_landscape = ibnot_api.make_sine_landscape
make_uniform = ibnot_api.make_uniform
NativeConfig = ibnot_api.NativeConfig
RenderConfig = ibnot_api.RenderConfig
OutputConfig = ibnot_api.OutputConfig
InferenceRequest = ibnot_api.InferenceRequest
run_inference = ibnot_api.run_inference

SIZE = 512
NUM_SITES = 1024
SEED = 7
OUTPUT_WIDTH = SIZE
OUTPUT_HEIGHT = SIZE
POINT_RADIUS = 0.004
DPI = 300
INVERT_DENSITY = True
NATIVE_TIMER = True
RENDER_ENABLED = True
PNG_ENABLED = True
KEEP_PGM = True
KEEP_EPS = True
KEEP_STATS = True
OUTPUT_DIR = REPO_ROOT / "python" / "notebooks" / "_generated_colab"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLI = REPO_ROOT / "ibnot_cli" / "build-colab" / "ibnot_new_cli"

if not CLI.exists():
    raise FileNotFoundError(f"CLI executable missing: {CLI}")
if shutil.which("gs") is None and PNG_ENABLED:
    raise RuntimeError("Ghostscript ('gs') is required for notebook PNG previews.")

def build_request(image, stem: str, *, invert: bool = INVERT_DENSITY):
    render = RenderConfig(
        enabled=RENDER_ENABLED,
        render_width=OUTPUT_WIDTH if RENDER_ENABLED else None,
        render_height=OUTPUT_HEIGHT if RENDER_ENABLED else None,
        point_radius=POINT_RADIUS,
        png_enabled=PNG_ENABLED,
        dpi=DPI,
    )
    native = NativeConfig(
        num_sites=NUM_SITES,
        seed=SEED,
        max_iters=25,
        max_newton_iters=50,
        invert=invert,
        native_timer=NATIVE_TIMER,
    )
    output = OutputConfig(
        output_dir=OUTPUT_DIR,
        output_stem=stem,
        keep_pgm=KEEP_PGM,
        keep_eps=KEEP_EPS,
        keep_stats_txt=KEEP_STATS,
    )
    return InferenceRequest(image_array=image, native=native, render=render, output=output, executable=CLI)

print(f"API path: {API_PATH}")
print(f"run_inference signature: {inspect.signature(run_inference)}")
print(f"CLI: {CLI}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Invert density: {INVERT_DENSITY}")

def show_case(title: str, image: np.ndarray, result):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title(f"{title} input")
    axes[0].axis("off")
    if result.png_path is not None:
        rendered = plt.imread(result.png_path)
        axes[1].imshow(rendered)
        axes[1].set_title(f"{title} output")
        axes[1].axis("off")
    else:
        axes[1].text(0.5, 0.5, "No PNG output", ha="center", va="center")
        axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print("stats:", result.stats)
    print("timings:", result.timings)
    if result.warnings:
        print("warnings:", result.warnings)


## 3. Uniform density example

Generate a constant `512x512` grayscale field with value `1.0`, build a structured inference request, run the native solver, and inspect the returned metadata, timings, and rendered result.


In [ ]:
uniform = make_uniform(size=SIZE, value=1.0)
uniform_result = run_inference(build_request(uniform, "uniform_512_1024"))
show_case("Uniform", uniform, uniform_result)


## 4. Left-to-right ramp example

Generate a linear ramp field, run it through the same structured request path, and compare the input image with the rendered output and timing metadata.


In [ ]:
ramp = make_linear_ramp(size=SIZE, left=1.0, right=0.0)
ramp_result = run_inference(build_request(ramp, "ramp_512_1024"))
show_case("Ramp", ramp, ramp_result)


## 5. 2D sine landscape example

Generate a normalized product-of-sines field, run it through the structured inference interface, and inspect the native statistics and timing breakdown.


In [ ]:
sine = make_sine_landscape(size=SIZE, fx=4, fy=4)
sine_result = run_inference(build_request(sine, "sine2d_512_1024"))
show_case("2D sine", sine, sine_result)


## 6. Upload your own image

Upload any image, convert it to grayscale with Pillow, resize it to the configured `SIZE x SIZE`, and run the same structured inference pipeline with the shared invert toggle.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No file uploaded.")

uploaded_name = next(iter(uploaded))
uploaded_path = Path(uploaded_name)
uploaded_image = Image.open(uploaded_path).convert("L")
uploaded_image = uploaded_image.resize((SIZE, SIZE))
uploaded_array = np.asarray(uploaded_image, dtype=float) / 255.0

uploaded_result = run_inference(build_request(uploaded_array, "uploaded_512_1024"))
show_case("Uploaded", uploaded_array, uploaded_result)
